In [ ]:
# -*- coding: utf-8 -*-
"""
光场强度交叉矩阵分析
基于光场仿真数据
"""
import os
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import pandas as pd

import torch
import numpy as np
import os
import time

from config import Config
from label_utils import evaluate_output
start_time = time.time()

# 设置随机种子，确保结果可重现
torch.manual_seed(42)
np.random.seed(42)

print("=" * 60)
print("多模式多波长光场调制系统 - 训练-仿真集成版 (Zero Padding)")
print("=" * 60)

# ===== 创建增强配置 =====

# 基本参数
num_modes = 3                                    # 模式数量
# wavelengths = np.array([1310e-9, 1550e-9])     # 波长列表(m)
wavelengths = np.array([1310e-9])     # 波长列表(m)
base_wavelength_idx = 0                       # 基准波长索引

# 空间参数
field_size = 50                                  # 场大小(像素)
layer_size = 300                                 # 层大小(像素)
focus_radius = 5                                 # 焦点半径(像素)
detectsize = 7                                  # 检测区域大小(像素)

# 物理参数
z_layers = 40e-6                                 # 层间距离(m)
z_prop = 150e-6                                  # 传播距离(m)
z_step = 20e-6                                   # 传播步长(m)
pixel_size = 1e-6                                # 像素大小(m)

# 检测区域偏移
offsets = [(0,0), (0,0)]                         # 每个波长的检测区域偏移

# 训练参数
learning_rate = 0.01                             # 学习率
lr_decay = 0.99                                  # 学习率衰减
epochs = 700                                     # 训练轮数
batch_size = 16                                  # 批量大小

# Zero Padding 参数
padding_ratio = 0.01                             # Padding 比例 (1%)
use_apodization = True                           # 启用边界衰减
apodization_width = 15                           # 衰减宽度

# MaskLoader 参数
fallback_focal_lengths = [40e-6, 60e-6, 80e-6, 100e-6, 120e-6]  # 备用掩码的焦距列表
default_num_layers = 3                           # 默认层数

# 保存参数
save_dir = f"./results/{num_modes}_mode_{len(wavelengths)}_wl_basewl_{wavelengths[base_wavelength_idx]}_z_prop_{z_prop}_focus_{focus_radius}/"
flag_savemat = True

# ===== 创建Config对象 =====
config = Config(
    num_modes=num_modes,
    wavelengths=wavelengths,
    base_wavelength_idx=base_wavelength_idx,
    field_size=field_size,
    layer_size=layer_size,
    focus_radius=focus_radius,
    detectsize=detectsize,
    z_layers=z_layers,
    z_prop=z_prop,
    z_step=z_step,
    pixel_size=pixel_size,
    offsets=offsets,
    learning_rate=learning_rate,
    lr_decay=lr_decay,
    epochs=epochs,
    batch_size=batch_size,
    padding_ratio=padding_ratio,
    use_apodization=use_apodization,
    apodization_width=apodization_width,
    fallback_focal_lengths=fallback_focal_lengths,
    default_num_layers=default_num_layers,
    save_dir=save_dir,
    flag_savemat=flag_savemat
)

print(f"✅ 配置创建成功！")
print(f"波长数量: {len(config.wavelengths)}")
print(f"模式数量: {config.num_modes}")
print(f"保存目录: {config.save_dir}")

def load_field_data(config):
    """加载光场数据"""
    
    print("🔍 加载光场数据...")
    
    # 查找所有.npy文件
    field_files = []
    base_dir = config.save_dir
    
    for file in os.listdir(base_dir):
        if file.endswith('.npy'):
            field_files.append(os.path.join(base_dir, file))
    
    print(f"✅ 找到 {len(field_files)} 个光场数据文件")
    
    # 按层数和模式组织数据
    field_data = defaultdict(lambda: defaultdict(list))
    
    for file_path in field_files:
        filename = os.path.basename(file_path)
        
        try:
            # 解析文件名获取层数和模式信息
            parts = filename.replace('.npy', '').split('_')
            
            mode_num = None
            layer_num = None
            
            for part in parts:
                if part.startswith('mode'):
                    mode_num = int(part.replace('mode', ''))
                elif part.endswith('layers'):
                    layer_num = int(part.replace('layers', ''))
            
            if mode_num is None or layer_num is None:
                print(f"❌ 跳过文件: {filename}")
                continue
            
            # 加载光场数据
            field = np.load(file_path)
            
            if field.size == 0:
                continue
            
            # 计算强度
            if np.iscomplexobj(field):
                intensity = np.abs(field)**2
            else:
                intensity = field**2
            
            field_data[layer_num][mode_num].append({
                'intensity': intensity,
                'field': field,
                'filename': filename,
                'layer': layer_num,
                'mode': mode_num,
                'max_intensity': np.max(intensity),
                'total_power': np.sum(intensity),
                'mean_intensity': np.mean(intensity)
            })
            
            print(f"  ✓ {layer_num}层 模式{mode_num}: {field.shape}, 最大强度: {np.max(intensity):.6f}")
            
        except Exception as e:
            print(f"❌ 处理失败 {filename}: {e}")
            continue
    
    return field_data

def plot_intensity_cross_matrices(normalized_matrices, visibility_results, all_layers, all_modes, save_dir):
    """绘制强度交叉矩阵"""
    
    print("\n🎨 绘制强度交叉矩阵...")
    
    os.makedirs(save_dir, exist_ok=True)
    
    num_layers = len(all_layers)
    
    # 🎨 统一字体大小设置
    TITLE_FONTSIZE = 24      # 标题字体
    LABEL_FONTSIZE = 24      # 轴标签字体
    TICK_FONTSIZE = 24       # 刻度标签字体
    VALUE_FONTSIZE = 24      # 数值标注字体
    
    if num_layers == 1:
        # 单个矩阵
        fig, ax = plt.subplots(1, 1, figsize=(10, 8))  # 🔧 增大图像尺寸
        axes = [ax]
    else:
        # 多个矩阵
        fig, axes = plt.subplots(1, num_layers, figsize=(8 * num_layers, 8), constrained_layout=True)  # 🔧 增大图像尺寸
        if num_layers == 1:
            axes = [axes]
    
    for idx, layer_num in enumerate(all_layers):
        if layer_num not in normalized_matrices:
            continue
            
        normalized_matrix = normalized_matrices[layer_num]
        visibility = visibility_results.get(layer_num, 0)
        
        ax = axes[idx]
        
        # 绘制热力图
        im = ax.imshow(normalized_matrix, cmap='Oranges', interpolation='nearest', vmin=0, vmax=1)
        
        # 🎨 统一字体大小
        ax.set_xlabel('Input Mode Index', fontsize=LABEL_FONTSIZE, fontweight='bold')
        ax.set_ylabel('Detector Regions', fontsize=LABEL_FONTSIZE, fontweight='bold')
        ax.set_title(f'{layer_num} Layers\n(Visibility: {visibility:.3f})', 
                    fontsize=TITLE_FONTSIZE, fontweight='bold')
        
        # 设置刻度标签
        ax.set_xticks(np.arange(len(all_modes)))
        ax.set_xticklabels([f'Mode{m}' for m in all_modes], fontsize=TICK_FONTSIZE, fontweight='bold')
        ax.set_yticks(np.arange(len(all_modes)))
        ax.set_yticklabels([f'Det{i+1}' for i in range(len(all_modes))], fontsize=TICK_FONTSIZE, fontweight='bold')
        
        # 添加数值标注
        for i in range(normalized_matrix.shape[0]):
            for j in range(normalized_matrix.shape[1]):
                value = normalized_matrix[i, j] * 100
                ax.text(j, i, f"{value:.1f}", ha='center', va='center', 
                       color='black', fontsize=VALUE_FONTSIZE, weight='bold')
    
    # 保存图像
    save_path = os.path.join(save_dir, 'intensity_cross_matrix.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')

    # 🔥 新增：每层单独保存
    for idx, layer_num in enumerate(all_layers):
        if layer_num not in normalized_matrices:
            continue
            
        # 为每层创建单独的图
        fig_single, ax_single = plt.subplots(1, 1, figsize=(10, 8))  # 🔧 增大单独图像尺寸
        
        normalized_matrix = normalized_matrices[layer_num]
        visibility = visibility_results.get(layer_num, 0)
        
        # 绘制单层热力图
        im_single = ax_single.imshow(normalized_matrix, cmap='Oranges', interpolation='nearest', vmin=0, vmax=1)
        
        # 🎨 统一字体大小
        ax_single.set_xlabel('Input Mode Index', fontsize=LABEL_FONTSIZE, fontweight='bold')
        ax_single.set_ylabel('Detector Regions', fontsize=LABEL_FONTSIZE, fontweight='bold')
        ax_single.set_title(f'{layer_num} Layers', 
                           fontsize=TITLE_FONTSIZE, fontweight='bold')
        
        # 设置刻度标签
        ax_single.set_xticks(np.arange(len(all_modes)))
        ax_single.set_xticklabels([f'Mode{m}' for m in all_modes], fontsize=TICK_FONTSIZE, fontweight='bold')
        ax_single.set_yticks(np.arange(len(all_modes)))
        ax_single.set_yticklabels([f'Det{i+1}' for i in range(len(all_modes))], fontsize=TICK_FONTSIZE, fontweight='bold')
        
        # 添加数值标注
        for i in range(normalized_matrix.shape[0]):
            for j in range(normalized_matrix.shape[1]):
                value = normalized_matrix[i, j] * 100
                ax_single.text(j, i, f"{value:.1f}", ha='center', va='center', 
                            color='black', fontsize=VALUE_FONTSIZE, weight='bold')
        
        plt.tight_layout()
        
        # 保存单层图像
        single_save_path = os.path.join(save_dir, f'intensity_cross_matrix_{layer_num}layers.png')
        plt.savefig(single_save_path, dpi=300, bbox_inches='tight')
        plt.close()
        
        print(f"  ✅ 单独保存: intensity_cross_matrix_{layer_num}layers.png")

    plt.close()  # 关闭原来的合成图

    
    print(f"  ✅ 保存: intensity_cross_matrix.png")
    
    # 绘制可见度对比图
    if len(visibility_results) > 1:
        plt.figure(figsize=(10, 8))  # 🔧 增大可见度图尺寸
        layers = list(visibility_results.keys())
        visibilities = list(visibility_results.values())
        
        plt.plot(layers, visibilities, marker='o', linestyle='-', color='orange', 
                linewidth=4, markersize=12)  # 🔧 增大线宽和标记尺寸
        plt.ylim(0, 1)
        
        # 🎨 统一字体大小
        plt.xlabel("Number of Layers", fontsize=LABEL_FONTSIZE, fontweight='bold')
        plt.ylabel("Visibility", fontsize=LABEL_FONTSIZE, fontweight='bold')
        plt.title("Visibility vs Number of Layers", fontsize=TITLE_FONTSIZE, fontweight='bold')
        plt.xticks(layers, fontsize=TICK_FONTSIZE, fontweight='bold')
        plt.yticks(fontsize=TICK_FONTSIZE, fontweight='bold')
        
        # 添加数值标注
        for x, y in zip(layers, visibilities):
            plt.text(x, y + 0.02, f"{y:.3f}", ha='center', va='bottom', 
                    fontsize=VALUE_FONTSIZE, weight='bold')
        
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        
        save_path = os.path.join(save_dir, 'visibility_comparison.png')
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
        
        print(f"  ✅ 保存: visibility_comparison.png")

def save_cross_matrix_data(cross_matrices, normalized_matrices, visibility_results, all_layers, all_modes, save_dir):
    """保存交叉矩阵数据"""
    
    print("\n💾 保存交叉矩阵数据...")
    
    # 保存为numpy文件
    for layer_num in all_layers:
        if layer_num in cross_matrices:
            # 原始强度矩阵
            np.save(os.path.join(save_dir, f'intensity_matrix_{layer_num}layers_raw.npy'), 
                   cross_matrices[layer_num])
            
            # 归一化矩阵
            np.save(os.path.join(save_dir, f'intensity_matrix_{layer_num}layers_normalized.npy'), 
                   normalized_matrices[layer_num])
    
    # 保存为CSV文件
    for layer_num in all_layers:
        if layer_num in normalized_matrices:
            df = pd.DataFrame(
                normalized_matrices[layer_num],
                index=[f'Detector{i+1}' for i in range(len(all_modes))],
                columns=[f'Mode{m}' for m in all_modes]
            )
            
            csv_path = os.path.join(save_dir, f'intensity_cross_matrix_{layer_num}layers.csv')
            df.to_csv(csv_path, encoding='utf-8-sig')
            
            print(f"  ✅ 保存: intensity_cross_matrix_{layer_num}layers.csv")
    
    # 保存可见度结果
    visibility_df = pd.DataFrame([
        {'Layers': layer, 'Visibility': vis} 
        for layer, vis in visibility_results.items()
    ])
    
    visibility_path = os.path.join(save_dir, 'visibility_results.csv')
    visibility_df.to_csv(visibility_path, index=False, encoding='utf-8-sig')
    
    print(f"  ✅ 保存: visibility_results.csv")

def print_cross_matrix_summary(cross_matrices, normalized_matrices, visibility_results):
    """打印交叉矩阵摘要"""
    
    print("\n📋 强度交叉矩阵摘要")
    print("="*50)
    
    print(f"分析配置数量: {len(cross_matrices)}")
    if cross_matrices:
        print(f"层数范围: {min(cross_matrices.keys())} - {max(cross_matrices.keys())}")
    
    print(f"\n各配置可见度:")
    print("-" * 30)
    
    for layer_num in sorted(visibility_results.keys()):
        visibility = visibility_results[layer_num]
        print(f"{layer_num:2d}层: {visibility:.4f}")
    
    # 找到最佳配置
    if visibility_results:
        best_layer = max(visibility_results.keys(), key=lambda k: visibility_results[k])
        best_visibility = visibility_results[best_layer]
        
        print(f"\n🏆 最佳配置: {best_layer}层")
        print(f"   最高可见度: {best_visibility:.4f}")
        
        # 显示最佳配置的矩阵
        if best_layer in normalized_matrices:
            print(f"\n最佳配置交叉矩阵 ({best_layer}层):")
            print("-" * 40)
            best_matrix = normalized_matrices[best_layer]
            
            # 打印表头
            print("       ", end="")
            for mode in range(best_matrix.shape[1]):
                print(f"Mode{mode+1:2d} ", end="")
            print()
            
            # 打印矩阵内容
            for det in range(best_matrix.shape[0]):
                print(f"Det{det+1:2d}: ", end="")
                for mode in range(best_matrix.shape[1]):
                    value = best_matrix[det, mode] * 100
                    print(f"{value:5.1f}% ", end="")
                print()

def visualize_individual_mode_regions(field_data, layer_num, evaluation_regions, save_dir=None):
    """为每个模式单独可视化检测区域"""
    
    print(f"   🎨 为第{layer_num}层的每个模式生成单独图片...")
    
    if layer_num not in field_data:
        print(f"   ❌ 第{layer_num}层无数据")
        return
    
    layer_data = field_data[layer_num]
    
    # 为每个模式单独生成图片
    for mode_idx in range(1, 4):  # 模式1, 2, 3
        if mode_idx not in layer_data or not layer_data[mode_idx]:
            print(f"   ⚠️ 模式{mode_idx}无数据")
            continue
        
        # 获取该模式的强度数据
        mode_data = layer_data[mode_idx][0]  # 取第一个样本
        intensity = mode_data['intensity']
        
        # 创建图片
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
        
        # 左图：该模式的纯强度分布
        im1 = ax1.imshow(intensity, cmap='hot', origin='upper')
        ax1.set_title(f"Layer {layer_num} - Mode {mode_idx} Intensity")
        ax1.axis('off')
        plt.colorbar(im1, ax=ax1, shrink=0.8)
        
        # 右图：该模式的强度 + 所有检测区域
        im2 = ax2.imshow(intensity, cmap='hot', origin='upper')
        ax2.set_title(f"Layer {layer_num} - Mode {mode_idx} + Detection Regions")
        ax2.axis('off')
        
        # 绘制所有检测区域
        colors = ['cyan', 'lime', 'yellow', 'magenta', 'orange']
        for i, (x_start, x_end, y_start, y_end) in enumerate(evaluation_regions):
            color = colors[i % len(colors)]
            
            # 绘制矩形框
            rect = plt.Rectangle((x_start, y_start), x_end - x_start, y_end - y_start,
                               linewidth=2, edgecolor=color, facecolor='none')
            ax2.add_patch(rect)
            
            # 添加标签
            center_x = (x_start + x_end) / 2
            center_y = (y_start + y_end) / 2
            ax2.text(center_x, center_y, f'Det{i+1}', 
                    ha='center', va='center', fontsize=10, 
                    color='white', weight='bold')
        
        plt.colorbar(im2, ax=ax2, shrink=0.8)
        plt.tight_layout()
        
        # 保存图片
        if save_dir:
            os.makedirs(save_dir, exist_ok=True)
            filename = f'layer_{layer_num}_mode_{mode_idx}_detection_regions.png'
            save_path = os.path.join(save_dir, filename)
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f"   ✅ 保存: {filename}")
        
        plt.close()  # 不显示图片，只保存

def visualize_all_modes_combined(field_data, layer_num, evaluation_regions, save_dir=None):
    """显示所有模式的合成图（原来的功能）"""
    
    print(f"   🎨 生成第{layer_num}层的合成图...")
    
    if layer_num not in field_data:
        return
    
    layer_data = field_data[layer_num]
    
    # 合成所有模式的强度
    combined_intensity = None
    valid_modes = 0
    
    for mode_idx in range(1, 4):
        if mode_idx in layer_data and layer_data[mode_idx]:
            mode_intensity = layer_data[mode_idx][0]['intensity']
            
            if combined_intensity is None:
                combined_intensity = np.zeros_like(mode_intensity)
            
            combined_intensity += mode_intensity
            valid_modes += 1
    
    if combined_intensity is None:
        print(f"   ❌ 第{layer_num}层无有效数据")
        return
    
    # 创建合成图
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # 左图：合成强度
    im1 = ax1.imshow(combined_intensity, cmap='hot', origin='upper')
    ax1.set_title(f"Layer {layer_num} - Combined Intensity ({valid_modes} modes)")
    ax1.axis('off')
    plt.colorbar(im1, ax=ax1, shrink=0.8)
    
    # 右图：合成强度 + 检测区域
    im2 = ax2.imshow(combined_intensity, cmap='hot', origin='upper')
    ax2.set_title(f"Layer {layer_num} - Combined + Detection Regions")
    ax2.axis('off')
    
    # 绘制检测区域
    colors = ['cyan', 'lime', 'yellow', 'magenta', 'orange']
    for i, (x_start, x_end, y_start, y_end) in enumerate(evaluation_regions):
        color = colors[i % len(colors)]
        
        rect = plt.Rectangle((x_start, y_start), x_end - x_start, y_end - y_start,
                           linewidth=2, edgecolor=color, facecolor='none')
        ax2.add_patch(rect)
        
        center_x = (x_start + x_end) / 2
        center_y = (y_start + y_end) / 2
        ax2.text(center_x, center_y, f'Det{i+1}', 
                ha='center', va='center', fontsize=10, 
                color='white', weight='bold')
    
    plt.colorbar(im2, ax=ax2, shrink=0.8)
    plt.tight_layout()
    
    # 保存合成图
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        filename = f'layer_{layer_num}_combined_detection_regions.png'
        save_path = os.path.join(save_dir, filename)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"   ✅ 保存合成图: {filename}")
    
    plt.close()


def fix_evaluation_regions(evaluation_regions):
    """修复评估区域坐标，确保都是整数"""
    return [(int(x_start), int(x_end), int(y_start), int(y_end)) 
            for x_start, x_end, y_start, y_end in evaluation_regions]


def debug_detection_regions(intensity, evaluation_regions, layer_num, mode_num):
    """调试检测区域与强度分布的匹配"""
    
    print(f"\n🔍 调试第{layer_num}层模式{mode_num}:")
    print(f"   强度图尺寸: {intensity.shape}")
    print(f"   最大强度位置: {np.unravel_index(np.argmax(intensity), intensity.shape)}")
    print(f"   最大强度值: {np.max(intensity):.6f}")
    
    # 检查每个检测区域
    for i, (x_start, x_end, y_start, y_end) in enumerate(evaluation_regions):
        # 提取区域强度
        region_intensity = intensity[y_start:y_end, x_start:x_end]
        region_sum = np.sum(region_intensity)
        region_max = np.max(region_intensity)
        
        print(f"   Det{i+1}: 区域({x_start}-{x_end}, {y_start}-{y_end})")
        print(f"          总强度: {region_sum:.6f}")
        print(f"          最大强度: {region_max:.6f}")
        print(f"          区域尺寸: {region_intensity.shape}")
        
        # 检查是否包含全局最大值
        max_y, max_x = np.unravel_index(np.argmax(intensity), intensity.shape)
        contains_max = (x_start <= max_x < x_end) and (y_start <= max_y < y_end)
        print(f"          包含最大值: {'✓' if contains_max else '✗'}")
def verify_coordinate_system(intensity, evaluation_regions):
    """验证坐标系统是否正确"""
    
    print(f"\n📐 坐标系统验证:")
    print(f"   图像形状: {intensity.shape} (height, width)")
    
    # 找到最大值位置
    max_pos = np.unravel_index(np.argmax(intensity), intensity.shape)
    max_y, max_x = max_pos
    print(f"   最大值位置: row={max_y}, col={max_x}")
    
    # 检查每个区域的坐标定义
    for i, (x_start, x_end, y_start, y_end) in enumerate(evaluation_regions):
        print(f"   Det{i+1}: x({x_start}-{x_end}), y({y_start}-{y_end})")
        
        # 验证坐标是否在有效范围内
        valid_x = 0 <= x_start < x_end <= intensity.shape[1]
        valid_y = 0 <= y_start < y_end <= intensity.shape[0]
        
        print(f"          坐标有效性: x={'✓' if valid_x else '✗'}, y={'✓' if valid_y else '✗'}")
        
        if valid_x and valid_y:
            # 检查区域是否包含最大值
            contains_max = (x_start <= max_x < x_end) and (y_start <= max_y < y_end)
            print(f"          包含光斑: {'✓' if contains_max else '✗'}")

def comprehensive_data_consistency_check(field_data, config):
    """全面的数据一致性检查"""
    
    print("\n🔍 数据一致性全面检查")
    print("="*50)
    
    for layer_num in sorted(field_data.keys()):
        print(f"\n📊 检查第{layer_num}层:")
        
        if layer_num not in field_data:
            continue
            
        layer_data = field_data[layer_num]
        
        # 获取样本数据
        sample_data = None
        for mode_data_list in layer_data.values():
            if mode_data_list:
                sample_data = mode_data_list[0]
                break
        
        if sample_data is None:
            continue
            
        field_size = sample_data['intensity'].shape[0]
        
        # 创建检测区域
        from data_generator import create_evaluation_regions_by_wavelength
        evaluation_regions_raw = create_evaluation_regions_by_wavelength(
            field_size, field_size, config.focus_radius, config.detectsize, 
            offsets=config.offsets, num_modes=config.num_modes
        )
        evaluation_regions = fix_evaluation_regions(evaluation_regions_raw)
        
        # 检查每个模式
        for mode_idx in range(1, 4):
            if mode_idx not in layer_data or not layer_data[mode_idx]:
                continue
                
            intensity = layer_data[mode_idx][0]['intensity']
            
            print(f"\n   模式{mode_idx}:")
            
            # 调试检测区域
            # debug_detection_regions(intensity, evaluation_regions, layer_num, mode_idx)
            
            # 验证坐标系统
            # verify_coordinate_system(intensity, evaluation_regions)
            
            # 计算实际的区域强度
            region_intensities = evaluate_output(intensity, evaluation_regions)
            print(f"   计算得到的区域强度: {region_intensities}")
            
            # 找出最强的检测区域
            max_det_idx = np.argmax(region_intensities)
            print(f"   最强检测区域: Det{max_det_idx+1} ({region_intensities[max_det_idx]:.6f})")

def calculate_intensity_cross_matrix_enhanced(field_data, config):
    """增强版交叉矩阵计算 - 带进度显示"""
    
    print("\n📊 计算强度交叉矩阵...")
    
    # 导入检测区域创建函数
    from data_generator import create_evaluation_regions_by_wavelength
    
    # 获取参数
    num_modes = config.num_modes
    focus_radius = config.focus_radius
    detectsize = config.detectsize
    
    # 获取所有层数和模式
    all_layers = sorted(field_data.keys())
    all_modes = sorted(set().union(*[layer_data.keys() for layer_data in field_data.values()]))
    
    print(f"📋 分析计划: {len(all_layers)}个层配置 × {len(all_modes)}个模式")
    print(f"层数: {all_layers}")
    print(f"模式: {all_modes}")
    
    # 结果存储
    cross_matrices = {}
    normalized_matrices = {}
    visibility_results = {}
    comprehensive_data_consistency_check(field_data, config)
    # 处理每个层配置
    for layer_idx, layer_num in enumerate(all_layers):
        print(f"\n🔄 [{layer_idx+1}/{len(all_layers)}] 处理 {layer_num} 层配置...")
        
        if layer_num not in field_data:
            print(f"⚠️ 跳过：无数据")
            continue
            
        layer_data = field_data[layer_num]
        
        try:
            # 创建交叉矩阵
            intensity_matrix = np.zeros((num_modes, num_modes))
            
            # 获取样本数据
            sample_data = None
            for mode_data_list in layer_data.values():
                if mode_data_list:
                    sample_data = mode_data_list[0]
                    break
            
            if sample_data is None:
                print(f"❌ 无有效数据")
                continue
                
            field_size = sample_data['intensity'].shape[0]
            
            # 创建检测区域
            evaluation_regions_raw = create_evaluation_regions_by_wavelength(
                field_size, field_size, focus_radius, detectsize, 
                offsets=config.offsets, num_modes=num_modes
            )
            evaluation_regions = fix_evaluation_regions(evaluation_regions_raw)

            print(f"   ✓ 检测区域: {len(evaluation_regions)}个")
            print("   📊 显示检测区域分布...")
            
            # 创建保存目录
            vis_save_dir = os.path.join(config.save_dir, "detection_visualization")
            
            # 生成每个模式的单独图片（这是您想要的）
            visualize_individual_mode_regions(field_data, layer_num, evaluation_regions, vis_save_dir)
            
            # 可选：也生成合成图
            visualize_all_modes_combined(field_data, layer_num, evaluation_regions, vis_save_dir)
            
            # 计算每个模式的强度分布
            processed_modes = 0
            # 更清晰的版本：
            for input_mode in all_modes:
                if input_mode not in layer_data:
                    continue
                    
                mode_data_list = layer_data[input_mode]
                if not mode_data_list:
                    continue
                
                # 计算区域强度
                total_intensities = []
                for data_entry in mode_data_list:
                    intensity = data_entry['intensity']
                    region_intensities = evaluate_output(intensity, evaluation_regions)
                    total_intensities.append(region_intensities)
                
                if total_intensities:
                    avg_intensities = np.mean(total_intensities, axis=0)
                    
                    # 正确的矩阵填充
                    mode_col_idx = input_mode - 1  # 模式1->列0, 模式2->列1, 模式3->列2
                    for detector_idx in range(min(len(avg_intensities), num_modes)):
                        intensity_matrix[detector_idx, mode_col_idx] = avg_intensities[detector_idx]
                    
                    processed_modes += 1
                    print(f"   ✓ 模式{input_mode}: 平均强度 {np.mean(avg_intensities):.6f}")

            
            # 保存和计算结果
            cross_matrices[layer_num] = intensity_matrix.copy()
            
            # 归一化
            normalized_matrix = np.zeros_like(intensity_matrix)
            for col in range(intensity_matrix.shape[1]):
                col_sum = np.sum(intensity_matrix[:, col])
                if col_sum > 0:
                    normalized_matrix[:, col] = intensity_matrix[:, col] / col_sum
            
            normalized_matrices[layer_num] = normalized_matrix
            
            # 计算可见度
            visibility = np.mean(np.diag(normalized_matrix))
            visibility_results[layer_num] = visibility
            
            print(f"   🎯 可见度: {visibility:.4f}")
            
        except Exception as e:
            print(f"   ❌ 处理失败: {e}")
            continue
    
    print(f"\n✅ 交叉矩阵计算完成！处理了 {len(cross_matrices)} 个配置")
    return cross_matrices, normalized_matrices, visibility_results, all_layers, all_modes

# ===== 完整的修复版主函数 =====
def main_intensity_cross_matrix_analysis(config):
    """主要的强度交叉矩阵分析函数 - 修复版本"""
    
    print("\n" + "="*60)
    print("光场强度交叉矩阵分析 - 修复版本")
    print("="*60)
    
    # 从 config 获取参数
    print(f"📋 使用配置参数:")
    print(f"   模式数量: {config.num_modes}")
    print(f"   焦点半径: {config.focus_radius}")
    print(f"   检测区域大小: {config.detectsize}")
    
    # 1. 加载数据
    field_data = load_field_data(config)
    
    if not field_data:
        print("❌ 未找到有效的光场数据")
        return None, None, None
    
    # 2. 计算交叉矩阵 - 使用修复版本
    cross_matrices, normalized_matrices, visibility_results, all_layers, all_modes = \
        calculate_intensity_cross_matrix_enhanced(field_data, config)
    
    if not cross_matrices:
        print("❌ 交叉矩阵计算失败")
        return None, None, None
    
    # 3. 创建保存目录
    save_dir = os.path.join(config.save_dir, "intensity_cross_matrix_analysis")
    os.makedirs(save_dir, exist_ok=True)
    
    # 4. 绘制矩阵
    plot_intensity_cross_matrices(normalized_matrices, visibility_results, all_layers, all_modes, save_dir)
    
    # 5. 保存数据
    save_cross_matrix_data(cross_matrices, normalized_matrices, visibility_results, all_layers, all_modes, save_dir)
    
    # 6. 打印摘要
    print_cross_matrix_summary(cross_matrices, normalized_matrices, visibility_results)
    
    print(f"\n🎉 强度交叉矩阵分析完成！")
    print(f"📁 结果保存在: {save_dir}")
    print(f"📊 查看交叉矩阵: intensity_cross_matrix.png")
    print(f"📈 查看可见度对比: visibility_comparison.png")
    print(f"📋 查看数据表: intensity_cross_matrix_*layers.csv")
    print(f"🎨 查看检测区域可视化: detection_visualization/")
    
    return cross_matrices, normalized_matrices, visibility_results

# ===== 在您的主程序最后添加 - 修复版本 =====
print("\n" + "="*60)
print("执行强度交叉矩阵分析")
print("="*60)

# 运行修复版本的分析
try:
    cross_matrices, normalized_matrices, visibility_results = main_intensity_cross_matrix_analysis(config)
    
    if cross_matrices is not None:
        print("✅ 强度交叉矩阵分析成功完成！")
        
        # 显示简要结果
        print("\n📊 可见度摘要:")
        for layer_num, visibility in visibility_results.items():
            print(f"   {layer_num}层: {visibility:.4f}")
        
        # 显示生成的文件总结
        print("\n📁 生成的文件:")
        print("   📊 交叉矩阵分析:")
        print("      - intensity_cross_matrix.png")
        print("      - visibility_comparison.png")
        print("      - intensity_cross_matrix_*layers.csv")
        print("      - visibility_results.csv")
        
        print("   🎨 检测区域可视化:")
        vis_dir = os.path.join(config.save_dir, "detection_visualization")
        if os.path.exists(vis_dir):
            png_files = [f for f in os.listdir(vis_dir) if f.endswith('.png')]
            for png_file in sorted(png_files):
                print(f"      - {png_file}")
        
    else:
        print("❌ 分析失败，请检查数据文件")
        
except Exception as e:
    print(f"❌ 分析过程中出现错误: {e}")
    import traceback
    traceback.print_exc()

print(f"\n⏱️ 总运行时间: {time.time() - start_time:.2f} 秒")
print("🎉 所有分析完成！")

多模式多波长光场调制系统 - 训练-仿真集成版 (Zero Padding)
已调整offsets数量以匹配波长数量: 1
✓ 基准波长设置: 索引 0 -> 1310.0nm
✓ Zero Padding 配置: 填充比例=0.01, 衰减=True
  - 衰减类型: cosine, 宽度: 15像素
配置完成，使用设备: cuda
✅ 配置创建成功！
波长数量: 1
模式数量: 3
保存目录: ./results/3_mode_1_wl_basewl_1.31e-06_z_prop_0.00015_focus_5/

执行强度交叉矩阵分析

光场强度交叉矩阵分析 - 修复版本
📋 使用配置参数:
   模式数量: 3
   焦点半径: 5
   检测区域大小: 7
🔍 加载光场数据...
✅ 找到 15 个光场数据文件
  ✓ 2层 模式3: (300, 300), 最大强度: 0.230135
  ✓ 3层 模式3: (300, 300), 最大强度: 0.270215
  ✓ 5层 模式2: (300, 300), 最大强度: 0.260688
  ✓ 3层 模式1: (300, 300), 最大强度: 0.177087
  ✓ 4层 模式2: (300, 300), 最大强度: 0.293304
  ✓ 5层 模式3: (300, 300), 最大强度: 0.204957
  ✓ 4层 模式1: (300, 300), 最大强度: 0.249878
  ✓ 5层 模式1: (300, 300), 最大强度: 0.171563
  ✓ 2层 模式1: (300, 300), 最大强度: 0.098024
  ✓ 1层 模式2: (300, 300), 最大强度: 0.074251
  ✓ 3层 模式2: (300, 300), 最大强度: 0.328494
  ✓ 2层 模式2: (300, 300), 最大强度: 0.301196
  ✓ 1层 模式1: (300, 300), 最大强度: 0.037742
  ✓ 1层 模式3: (300, 300), 最大强度: 0.029310
  ✓ 4层 模式3: (300, 300), 最大强度: 0.247507

📊 计算强度交叉矩阵...
📋 分析计划: 5个层配置 × 3个模式
层数: [1, 2, 